# 다중 클래스 분류
- iris 데이터 셋으로 다중 클래스 분류
- 손실함수 CCE
- 최종예측 argmax()

### 1. 데이터 불러오기

In [72]:
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris()

X = pd.DataFrame(
    iris.data,
    columns= iris.feature_names
)

y = pd.Series(
    iris.target,
    name = 'target'
)

print(X.shape)
print(X.value_counts().sort_index())
print(iris.target_names)

(150, 4)
sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
4.3                3.0               1.1                0.1                 1
4.4                2.9               1.4                0.2                 1
                   3.0               1.3                0.2                 1
                   3.2               1.3                0.2                 1
4.5                2.3               1.3                0.3                 1
                                                                           ..
7.7                2.6               6.9                2.3                 1
                   2.8               6.7                2.0                 1
                   3.0               6.1                2.3                 1
                   3.8               6.7                2.2                 1
7.9                3.8               6.4                2.0                 1
Name: count, Length: 149, dtype: int64
['setosa' 'versicolor

### 2. 학습 테스트 데이터 분리

In [73]:
from sklearn.model_selection import train_test_split


X_train,X_test,y_train,y_test = train_test_split(
    X, y,
    test_size=0.2,     
    random_state=42,  
    stratify= y
)
print((X_train.shape),(X_test.shape),(y_train.shape),(y_test.shape))

(120, 4) (30, 4) (120,) (30,)


### 3. 데이터 표준화

In [74]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled

array([[-1.72156775, -0.33210111, -1.34572231, -1.32327558],
       [-1.12449223, -1.22765467,  0.41450518,  0.6517626 ],
       [ 1.14439475, -0.5559895 ,  0.58484978,  0.25675496],
       [-1.12449223,  0.11567567, -1.28894078, -1.45494479],
       [-0.40800161, -1.22765467,  0.13059752,  0.12508575],
       [ 0.54731923, -1.22765467,  0.69841284,  0.91510102],
       [-0.2885865 , -0.77987789,  0.24416059,  0.12508575],
       [ 0.54731923, -0.5559895 ,  0.75519438,  0.38842418],
       [ 2.21913069, -0.10821272,  1.3230097 ,  1.44177787],
       [ 2.21913069,  1.6828944 ,  1.66369889,  1.31010866],
       [ 2.09971558, -0.10821272,  1.60691736,  1.17843945],
       [ 0.18907392, -0.33210111,  0.41450518,  0.38842418],
       [-1.00507713, -2.34709662, -0.15331014, -0.26992188],
       [-0.04975629, -0.77987789,  0.18737906, -0.26992188],
       [-0.04975629, -1.00376628,  0.13059752, -0.00658346],
       [-1.36332244,  0.33956406, -1.23215924, -1.32327558],
       [-0.88566202,  1.

In [75]:
X_test_scaled

array([[-1.72156775, -0.10821272, -1.40250384, -1.32327558],
       [ 0.30848902, -0.10821272,  0.64163131,  0.78343181],
       [-1.12449223, -1.45154306, -0.2668732 , -0.26992188],
       [-1.00507713, -1.67543145, -0.2668732 , -0.26992188],
       [-1.72156775,  0.33956406, -1.40250384, -1.32327558],
       [ 0.54731923,  0.56345245,  0.52806825,  0.52009339],
       [-1.48273754,  1.23511762, -1.57284844, -1.32327558],
       [-0.52741671,  0.78734084, -1.17537771, -1.32327558],
       [ 0.78614944, -0.10821272,  0.81197591,  1.04677024],
       [-0.52741671, -0.10821272,  0.41450518,  0.38842418],
       [ 1.74147027, -0.33210111,  1.43657276,  0.78343181],
       [ 1.26380985,  0.11567567,  0.75519438,  1.44177787],
       [ 0.78614944, -0.10821272,  1.1526651 ,  1.31010866],
       [ 0.66673433,  0.33956406,  0.41450518,  0.38842418],
       [-1.00507713,  0.78734084, -1.28894078, -1.32327558],
       [-1.00507713,  0.56345245, -1.34572231, -1.32327558],
       [-0.04975629,  2.

### 4. 텐서 변환

In [76]:
import torch

# 입력 데이터는 PyTorch Tensor float32 타입 사용
X_train_tensor = torch.tensor(X_train_scaled, dtype = torch.float32)    # 
X_test_tensor = torch.tensor(X_test_scaled, dtype = torch.float32)

# 분류 문제에서 CrossEntropyLoss 사용시 정답 라벨은 long타입을 요구 (정수형)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype = torch.long)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype = torch.long)

print(X_train_tensor.size())
print(y_train_tensor.size())
print(X_test_tensor.size())
print(y_test_tensor.size())

torch.Size([120, 4])
torch.Size([120])
torch.Size([30, 4])
torch.Size([30])


### 5. 모델 정의

In [77]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(X_train_tensor.size(-1), 16), # 입력특성 4 -> 은닉층 노드 16개
    nn.ReLU(),  # 비선형성 추가

    nn.Linear(16, 8),   # 16 은닉 -> 8 은닉
    nn.ReLU(),

    nn.Linear(8,3),     # 8 은닉 -> 최종 출력 3개 클래스
)

model

Sequential(
  (0): Linear(in_features=4, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=3, bias=True)
)

출력값 예시 : [0.2, 0.5, 0.3]   
가장 큰 값의 인덱스 : 1   
최종 예측 클래스 : versicolor

In [78]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss() # 다중 분류 손실 함수

optimizer = optim.Adam(
    model.parameters(), # 파라미터 업데이트
    lr=0.01
)



CrossEntropyLoss() : 모델의 출력값과 실제 클래스 번호를 비교해서 다중 클래스 분류 손실 계산   
CrossEntropyLoss()를 사용시에는 Softmax()를 따로 출력층에 붙이지 않는다.

In [79]:
for epoch in range(1000):
    model.train()   # 학습 모드로 설정 

    optimizer.zero_grad()   # 이전 스텝 기울기 초기화
    output = model(X_train_tensor)  # 순전파 : 결과 예측
    loss = criterion(output, y_train_tensor)    # 손실 계산

    loss.backward() # 기울기 계산
    optimizer.step()    # 가중치 업데이트

    if (epoch + 1) %100 ==0:
        print(f"Epoch{epoch + 1}의 Loss : {loss.item():.4f}")

Epoch100의 Loss : 0.0383
Epoch200의 Loss : 0.0268
Epoch300의 Loss : 0.0227
Epoch400의 Loss : 0.0216
Epoch500의 Loss : 0.0212
Epoch600의 Loss : 0.0209
Epoch700의 Loss : 0.0207
Epoch800의 Loss : 0.0206
Epoch900의 Loss : 0.0205
Epoch1000의 Loss : 0.0203


### 8. 출력값 확인

In [80]:
model.eval()    # 평가모드로 전환

with torch.no_grad():
    output = model(X_test_tensor)   # 테스트 데이터로 예측값 계산


print(output[:5])   # 확률이 아닌 클래스별 원시 점수(로직) 출력
print(output.shape)

tensor([[ 10.9890,  -1.5557, -15.7407],
        [ -3.6415,   5.2605,   6.2399],
        [ -0.4677,  11.5736,  -8.2409],
        [ -1.0541,  13.1395,  -8.1543],
        [ 10.7748,  -3.1671, -12.7226]])
torch.Size([30, 3])


### 9. 클래스 예측

In [81]:
with torch.no_grad():
    output = model(X_test_tensor)
    prediction = output.argmax(dim=1)   # 가장 높은 점수의 클래스

print(output[:10])
print(prediction[:10])

tensor([[ 10.9890,  -1.5557, -15.7407],
        [ -3.6415,   5.2605,   6.2399],
        [ -0.4677,  11.5736,  -8.2409],
        [ -1.0541,  13.1395,  -8.1543],
        [ 10.7748,  -3.1671, -12.7226],
        [ -2.6206,   6.9118,  -0.6632],
        [ 11.3535,  -6.2505,  -9.0759],
        [ 10.6743,  -0.9902, -16.1485],
        [ -3.7784,  -1.6473,  19.3305],
        [ -1.7882,   7.5518,  -2.9785]])
tensor([0, 2, 1, 1, 0, 1, 0, 0, 2, 1])


### 10. 확률값 확인

In [83]:
with torch.no_grad():
    output = model(X_test_tensor)
    probability = torch.softmax(output , dim = 1)# 클래스별 확률 변환
    prediction = output.argmax(dim=1)   # 가장 높은 점수의 클래스

print(output[:10]) # 클래스별 로짓값
# 값이 여러개인 Tensor에서 하나의 값씩 접근하여 확인
for row in probability[:10]:
    print([f"{p.item():6f}" for p in row])  # 클래스별 확률값
print(prediction[:10])  # 예측 클래스

tensor([[ 10.9890,  -1.5557, -15.7407],
        [ -3.6415,   5.2605,   6.2399],
        [ -0.4677,  11.5736,  -8.2409],
        [ -1.0541,  13.1395,  -8.1543],
        [ 10.7748,  -3.1671, -12.7226],
        [ -2.6206,   6.9118,  -0.6632],
        [ 11.3535,  -6.2505,  -9.0759],
        [ 10.6743,  -0.9902, -16.1485],
        [ -3.7784,  -1.6473,  19.3305],
        [ -1.7882,   7.5518,  -2.9785]])
['0.999996', '0.000004', '0.000000']
['0.000037', '0.272993', '0.726969']
['0.000006', '0.999994', '0.000000']
['0.000001', '0.999999', '0.000000']
['0.999999', '0.000001', '0.000000']
['0.000072', '0.999415', '0.000513']
['1.000000', '0.000000', '0.000000']
['0.999991', '0.000009', '0.000000']
['0.000000', '0.000000', '1.000000']
['0.000088', '0.999885', '0.000027']
tensor([0, 2, 1, 1, 0, 1, 0, 0, 2, 1])


모델에 Softmax()함수는 사용하지 않는다.  
왜냐하면 CrossEntropyLoss에서 내부적으로 계산하기 때문에

### 11. 모델 평가

In [87]:
from sklearn.metrics import accuracy_score, confusion_matrix,classification_report

y_pred = prediction.numpy()
y_true = y_test_tensor.numpy()


print(f"정확도 : {accuracy_score(y_pred,y_true)}")
print(f"{confusion_matrix(y_true,y_pred)}")
print(f"{classification_report(y_true,y_pred,target_names= iris.target_names)}")

정확도 : 0.9333333333333333
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

